# MP0486 · RA1 · Tema 03 — Ejemplos de clase: procesamiento de datos con Pandas

**Propósito.** Resolver problemas breves con ventas ficticias para ilustrar la teoría de Pandas. Son ejemplos guiados, no una práctica evaluable. Ejecuta las celdas en orden; el cuaderno crea sus propios ficheros en una carpeta temporal de Colab. No necesita internet, instalaciones ni datos personales.

Antes de ejecutar cada bloque, intenta predecir su resultado; después contrasta tu explicación con los datos impresos. Los comentarios del código explican las decisiones importantes.

## 1 · Preparar una fuente real de ventas
**Problema.** Recibimos un CSV con un importe inválido y un dato ausente; además, un JSON con las categorías de los productos. Queremos reproducir el caso sin descargar archivos.
**Conceptos.** `Path` construye rutas, `csv` escribe un formato tabular y `json` escribe registros estructurados. Aquí solo **preparamos** entradas; Pandas las leerá en el ejemplo siguiente.
**Comenta:** ¿por qué conviene conservar un dato bruto inválido para probar la limpieza?

In [ ]:
from pathlib import Path
import csv
import json
import pandas as pd

datos = Path('tema03_datos')
datos.mkdir(exist_ok=True)  # Repetir la celda no falla si la carpeta ya existe.
ventas_csv = datos / 'ventas.csv'
productos_json = datos / 'productos.json'
filas = [
    {'id_venta': 1, 'id_producto': 101, 'ciudad': 'Barcelona', 'unidades': '2', 'precio': '12.50'},
    {'id_venta': 2, 'id_producto': 102, 'ciudad': 'Girona', 'unidades': '1', 'precio': '20.00'},
    {'id_venta': 3, 'id_producto': 101, 'ciudad': 'Barcelona', 'unidades': '3', 'precio': 'N/A'},
    {'id_venta': 4, 'id_producto': 103, 'ciudad': 'Lleida', 'unidades': '2', 'precio': '8.00'},
    {'id_venta': 5, 'id_producto': 999, 'ciudad': '', 'unidades': '1', 'precio': '10.00'},
]
with ventas_csv.open('w', encoding='utf-8', newline='') as fichero:
    escritor = csv.DictWriter(fichero, fieldnames=list(filas[0]))
    escritor.writeheader()  # Sin cabecera, Pandas no conocería los nombres de columna.
    escritor.writerows(filas)
productos = [
    {'id_producto': 101, 'categoria': 'Papelería'},
    {'id_producto': 102, 'categoria': 'Papelería'},
    {'id_producto': 103, 'categoria': 'Accesorios'},
]
productos_json.write_text(json.dumps(productos, ensure_ascii=False, indent=2), encoding='utf-8')
print('Fuentes creadas:', ventas_csv, productos_json)

## 2 · Inspeccionar sin dar por buenos los tipos
**Problema.** Antes de calcular importes necesitamos saber cuántas ventas hay, si hay valores ausentes y qué tipos ha inferido Pandas.
**Conceptos.** `read_csv` genera un `DataFrame`; `shape`, `head`, `dtypes` e `isna().sum()` permiten una primera auditoría. Una columna numérica puede convertirse en `float` por valores nulos: no presupongas su tipo.
**Comenta:** ¿cuántas filas tienen ciudad ausente? ¿Qué tipo presenta `precio` y por qué?

In [ ]:
brutas = pd.read_csv(ventas_csv, encoding='utf-8')  # Una fila CSV pasa a ser una fila del DataFrame.
print('Tamaño (filas, columnas):', brutas.shape)
print(brutas.head().to_string(index=False))
print('Tipos inferidos:', brutas.dtypes.astype(str).to_dict())
print('Ausentes por columna:', brutas.isna().sum().to_dict())

## 3 · Limpiar con una regla de negocio explícita
**Problema.** No se puede calcular una venta sin ciudad o precio válido. Queremos identificar las filas descartadas y obtener `importe = unidades × precio`.
**Conceptos.** `to_numeric(errors='coerce')` convierte valores ilegibles en `NaN`; `dropna(subset=...)` descarta únicamente filas con campos esenciales ausentes; las operaciones entre columnas son vectorizadas. Conservamos el origen en `brutas` para poder auditar qué cambió.
**Comenta:** ¿por qué es peligroso sustituir cualquier precio inválido por cero? ¿Cuántas filas válidas quedan?

In [ ]:
limpieza = brutas.copy()  # No modificamos la copia de datos de entrada.
limpieza['precio'] = pd.to_numeric(limpieza['precio'], errors='coerce')
limpieza['unidades'] = pd.to_numeric(limpieza['unidades'], errors='coerce')
obligatorios = ['ciudad', 'precio', 'unidades']
descartadas = limpieza[limpieza[obligatorios].isna().any(axis=1)]
validas = limpieza.dropna(subset=obligatorios).copy()
validas['importe'] = validas['unidades'] * validas['precio']  # Sin bucle fila a fila.
print('Descartadas:', descartadas['id_venta'].tolist())
print(validas[['id_venta', 'ciudad', 'importe']].to_string(index=False))
assert validas['importe'].tolist() == [25.0, 20.0, 16.0]

## 4 · Responder una pregunta filtrando filas
**Problema.** Marketing pide las ventas válidas de Barcelona cuyo importe sea superior a 20 €. Queremos solo las columnas útiles.
**Conceptos.** La máscara booleana se construye entre paréntesis; `&` combina condiciones **fila a fila** y `.loc[filas, columnas]` selecciona el resultado. En Pandas no usamos `and` para Series.
**Comenta:** ¿entraría una venta de importe exactamente 20? ¿Qué cambiarías para incluirla?

In [ ]:
mascara = (validas['ciudad'] == 'Barcelona') & (validas['importe'] > 20)
seleccion = validas.loc[mascara, ['id_venta', 'ciudad', 'importe']]
print(seleccion.to_string(index=False))
assert seleccion['id_venta'].tolist() == [1]

## 5 · Cruzar ventas con el catálogo de productos
**Problema.** El CSV no indica a qué categoría pertenece cada producto. La tabla JSON sí; debemos incorporarla sin duplicar ventas inesperadamente.
**Conceptos.** `read_json` crea otro `DataFrame`; `merge(..., on=..., how='left')` preserva las ventas del lado izquierdo; `validate='many_to_one'` exige que el catálogo tenga como máximo una fila por producto; `indicator=True` permite detectar productos sin correspondencia.
**Comenta:** ¿qué pasaría si hubiera dos entradas con `id_producto=101` en el catálogo? ¿Qué informa `_merge` sobre una clave desconocida?

In [ ]:
catalogo = pd.read_json(productos_json)
cruce = validas.merge(catalogo, on='id_producto', how='left', validate='many_to_one', indicator=True)
print(cruce[['id_venta', 'id_producto', 'categoria', '_merge']].to_string(index=False))
assert len(cruce) == len(validas)  # El cruce no ha multiplicado ventas.
assert cruce['_merge'].eq('both').all()

## 6 · Resumir las ventas por categoría
**Problema.** Dirección solicita el número de ventas, importe total e importe medio por categoría. Calcularemos tres medidas diferentes sobre cada grupo.
**Conceptos.** `groupby` forma grupos según una columna; `agg` asigna una función a cada medida; `reset_index()` convierte la clave agrupada en columna normal. No confundas recuento de ventas con suma de unidades.
**Comenta:** ¿por qué `Papelería` suma 45 € aunque sean dos ventas? ¿Cuánto sale la media?

In [ ]:
resumen = (cruce.groupby('categoria', as_index=False)
           .agg(numero_ventas=('id_venta', 'count'),
                importe_total=('importe', 'sum'),
                importe_medio=('importe', 'mean')))
print(resumen.to_string(index=False))
papeleria = resumen.loc[resumen['categoria'] == 'Papelería'].iloc[0]
assert (papeleria['numero_ventas'], papeleria['importe_total'], papeleria['importe_medio']) == (2, 45.0, 22.5)

## 7 · Comparar ciudades y categorías
**Problema.** Necesitamos una matriz con ciudades en filas, categorías en columnas e importes totales en cada cruce. Una celda sin ventas debería mostrar cero, no simular un registro nuevo.
**Conceptos.** `pivot_table` agrupa dos dimensiones; `aggfunc='sum'` suma importes; `fill_value=0` hace legible la ausencia de ventas en el resumen. `sort_values` organiza una vista, no modifica la fuente.
**Comenta:** ¿qué indica el cero de Girona en Accesorios? ¿Significa que hubo una venta de importe cero?

In [ ]:
matriz = pd.pivot_table(cruce, index='ciudad', columns='categoria', values='importe', aggfunc='sum', fill_value=0)
print(matriz.to_string())
print('Categorías por facturación:', resumen.sort_values('importe_total', ascending=False)['categoria'].tolist())

## 8 · Comunicar un resultado con un gráfico sencillo
**Problema.** Queremos mostrar a los compañeros cuánto factura cada categoría sin presentarles toda la tabla.
**Conceptos.** `.plot.bar` usa Matplotlib a través de Pandas; el gráfico representa **importes totales** y no medias. Ejes y título aclaran la magnitud. Un gráfico no sustituye la comprobación de los números.
**Comenta:** si tuvieras cientos de categorías, ¿seguiría siendo este gráfico legible?

In [ ]:
import matplotlib.pyplot as plt
ax = resumen.plot.bar(x='categoria', y='importe_total', legend=False, color='#34699a', figsize=(6, 3))
ax.set(title='Facturación por categoría', xlabel='Categoría', ylabel='Importe total (€)')
plt.xticks(rotation=0)
plt.tight_layout()  # Evita cortar etiquetas al visualizar o guardar.
plt.show()

## 9 · Entregar CSV y JSON y comprobarlos
**Problema.** Otro equipo pide el resumen en CSV y JSON. No basta con exportarlo: hay que releer los resultados y verificar que conservan las dos categorías y el importe esperado.
**Conceptos.** `to_csv(index=False)` evita una columna índice accidental; `to_json(orient='records')` produce una lista de objetos. `read_csv` y `read_json` permiten un **roundtrip** sencillo. Los tipos y la precisión pueden variar entre formatos: comprueba los campos esenciales.
**Comenta:** ¿qué pasaría si omites `index=False`? ¿Un archivo JSON con otra orientación tendría la misma estructura?

In [ ]:
salida_csv = datos / 'resumen.csv'
salida_json = datos / 'resumen.json'
resumen.to_csv(salida_csv, index=False, encoding='utf-8')
resumen.to_json(salida_json, orient='records', force_ascii=False, indent=2)
recuperado_csv = pd.read_csv(salida_csv)
recuperado_json = pd.read_json(salida_json, orient='records')
assert len(recuperado_csv) == len(recuperado_json) == 2
assert recuperado_csv['importe_total'].sum() == recuperado_json['importe_total'].sum() == 61.0
print('Entregables verificados:', salida_csv, salida_json)

## 10 · Reconocer otras opciones de lectura sin confundir formatos
**Problema.** Otro proveedor entrega XML o grandes CSV. Tenemos que reconocer las herramientas y sus límites, aunque este ejemplo se mantenga pequeño.
**Conceptos.** `read_xml` lee nodos repetidos como filas; cada XML debe indicar cuál es el nodo de registro. `read_csv(chunksize=...)` devuelve lotes para no cargar todo en memoria. Para Excel o Parquet existen `read_excel` y `read_parquet`, pero pueden requerir dependencias adicionales. No transformamos por arte de magia estructuras XML jerárquicas arbitrarias en tablas perfectas.
**Comenta:** si sumas importes por lotes, ¿por qué no debes sumar directamente las medias de cada lote?

In [ ]:
xml_ventas = datos / 'ventas_validas.xml'
cruce[['id_venta', 'ciudad', 'importe']].to_xml(xml_ventas, index=False, root_name='ventas', row_name='venta')
leido_xml = pd.read_xml(xml_ventas, xpath='./venta')  # Cada nodo venta es una fila.
total_por_lotes = 0.0
for lote in pd.read_csv(salida_csv, chunksize=1):  # Demostración con lotes pequeños, no necesidad real aquí.
    total_por_lotes += lote['importe_total'].sum()
assert leido_xml['importe'].sum() == 61.0
assert total_por_lotes == 61.0
print('XML recuperado:', len(leido_xml), 'ventas; total CSV por lotes:', total_por_lotes)

## Cierre
Hemos leído CSV y JSON, inspeccionado tipos y ausentes, convertido y filtrado, cruzado tablas, agrupado, construido una tabla dinámica, visualizado, exportado CSV/JSON/XML y comprobado resultados. Las entradas son ficticias y temporales. Para XML complejos, datos masivos o trabajo en equipo se necesitan decisiones y pruebas adicionales.

Las prácticas de desafío plantearán un problema **nuevo**, con elección de fuentes y herramientas; se podrá usar IA, pero tendrás que justificar limpieza, cálculos y salidas y resolver una validación individual. Selenium y Streamlit no se presuponen aprendidos en estos ejemplos: corresponden al autoaprendizaje guiado de la práctica posterior.